<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/Tesis_v4_Estilizada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background-color: #1a3e59; padding: 20px; border-radius: 10px;">
<h1 style="color: #ffffff; margin: 0; text-align: center;">Tesis: Análisis MCA de Deudores BCRA vs Padrón ARCA 🚀</h1>
<p style="color: #d1e8ff; text-align: center; margin-top: 10px; font-size: 1.1em;">Fase 2: Reducción de Dimensionalidad, Clustering de Entidades Financieras y Perfiles Demográficos.</p>
</div>


## <span style="color: #2b7a78;">1. Instalación de Librerías y Configuración Inicial ⚙️</span>


In [6]:
!pip install prince plotly polars pandas numpy kaleido
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import prince
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## <span style="color: #2b7a78;">2. Carga de Bases de Datos y Limpieza Poblacional 🧹</span>
En esta sección cargamos la Central de Deudores del BCRA y el padrón físico de ARCA. Aplicamos reglas de calidad de datos, unificamos la estructura de los CUIT/CUIL con `zfill()` e imprimimos métricas de resumen.


In [7]:
# ==========================================
# ⚙️ PASO 1: FIJANDO CEROS A LA IZQUIERDA (ZFILL) Y SUMMARY METRICS
# ==========================================
print("1. Cargando bases de datos en modo Lazy...")
df_bcra = pl.scan_parquet("/content/drive/MyDrive/Tesis2026/deudores_enero_2026_clasificados.parquet")
padron = pl.scan_parquet("/content/drive/MyDrive/Tesis2026/padron_fisicas.parquet") # Ajustado de _enero a padron normal si aplica
entidades_maestro = pl.scan_csv("/content/drive/MyDrive/Tesis2026/EntidadesFinancierasBCRA.csv")

print("Estandarizando estructuras de IDs y Códigos con ceros a la izquierda...")
# 1.1 Blindar el BCRA: forzar strings limpios y aplicar zfill reglamentario
df_bcra = df_bcra.with_columns([
    pl.col("nro_id").cast(pl.Utf8).str.strip_chars().str.zfill(11),
    pl.col("cod_entidad").cast(pl.Utf8).str.strip_chars().str.zfill(5)
])

# 1.2 Blindar el Padrón de ARCA: CUITs obligatorios de 11 caracteres
padron = padron.with_columns([
    pl.col("cuit").cast(pl.Utf8).str.strip_chars().str.zfill(11),
    pl.col("fecha_fallecimiento").str.strip_chars().alias("fallecimiento_clean"),
    pl.col("fecha_nacimiento").str.strip_chars().alias("nacimiento_clean"),
    pl.col("sexo").str.strip_chars().alias("sexo_clean"),
    pl.col("provincia").cast(pl.Utf8).str.zfill(2) # Preparando para las macro-regiones
])

# 1.3 Blindar el Maestro de Entidades: Códigos obligatorios de 5 caracteres
entidades_maestro = entidades_maestro.with_columns(
    pl.col("cod_entidad").cast(pl.Utf8).str.strip_chars().str.zfill(5)
)

print("Ejecutando cruce poblacional (BCRA + Padrón)...")
# Usamos LEFT JOIN para el diagnóstico inicial de pérdidas
joined = df_bcra.join(padron, left_on="nro_id", right_on="cuit", how="left")

print("\n" + "="*55)
print("📊 SUMMARY METRICS: PASO 1 (CRUCE Y POBLACIÓN PURIFICADA)")
print("="*55)

# Extraer métricas de control
metricas_join = joined.select([
    pl.len().alias("total_original"),
    pl.col("sexo_clean").is_not_null().sum().alias("conservados_join"),
    pl.col("sexo_clean").is_null().sum().alias("excluidos_no_padron"),
    (pl.col("nacimiento_clean") == "1901-01-01").sum().alias("fechas_inconsistentes"),
    ((pl.col("fallecimiento_clean") != "") & pl.col("fallecimiento_clean").is_not_null()).sum().alias("fallecidos_detectados"),
    (pl.col("sexo_clean") == "").sum().alias("sexo_vacio")
]).collect().to_pandas()

print(f"-> Total de registros originales BCRA: {metricas_join['total_original'][0]:,}")
print(f"-> Excluidos (Empresas / Sin Match de CUIT): {metricas_join['excluidos_no_padron'][0]:,}")
print(f"-> Conservados tras recuperar ceros iniciales: {metricas_join['conservados_join'][0]:,}")

print("\nAnomalías depuradas en los registros con match (Reglas ARCA):")
print(f"  - Fechas de nacimiento '1901-01-01' excluidas: {metricas_join['fechas_inconsistentes'][0]:,}")
print(f"  - Registros de personas fallecidas excluidos: {metricas_join['fallecidos_detectados'][0]:,}")
print(f"  - Género vacío excluido: {metricas_join['sexo_vacio'][0]:,}")

# Filtros sanitarios finales para el MCA
base_limpia = joined.filter(
    pl.col("sexo_clean").is_not_null()
).filter(
    (pl.col("fallecimiento_clean") == "") | pl.col("fallecimiento_clean").is_null()
).filter(
    pl.col("nacimiento_clean") != "1901-01-01"
).filter(
    pl.col("sexo_clean") != ""
)

total_final = base_limpia.select(pl.len()).collect().item()
print(f"\n=> 🏆 POBLACIÓN ACTIVA FINAL CONSERVADA POST-ZFILL: {total_final:,} deudores físicos")
print("="*55 + "\n")

# Hacemos el Join definitivo con el maestro de nombres, ahora que ambos son de 5 dígitos estrictos
joined_con_nombres = base_limpia.join(entidades_maestro, on="cod_entidad", how="left")



1. Cargando bases de datos en modo Lazy...
Estandarizando estructuras de IDs y Códigos con ceros a la izquierda...
Ejecutando cruce poblacional (BCRA + Padrón)...

📊 SUMMARY METRICS: PASO 1 (CRUCE Y POBLACIÓN PURIFICADA)
-> Total de registros originales BCRA: 32,642,267
-> Excluidos (Empresas / Sin Match de CUIT): 28,338,304
-> Conservados tras recuperar ceros iniciales: 4,303,963

Anomalías depuradas en los registros con match (Reglas ARCA):
  - Fechas de nacimiento '1901-01-01' excluidas: 7,477
  - Registros de personas fallecidas excluidos: 13,500
  - Género vacío excluido: 39

=> 🏆 POBLACIÓN ACTIVA FINAL CONSERVADA POST-ZFILL: 4,282,958 deudores físicos



## <span style="color: #2b7a78;">3. Filtrado Top 20 y Macro-Regiones Geográficas 🗺️</span>
Extraemos las 20 entidades con mayor volumen de deuda y segmentamos a los deudores en regiones macro utilizando el estándar del INDEC (AMBA, Región Pampeana, NOA, NEA, Cuyo, Patagonia).


In [8]:
# ==========================================
# 🏦 PASO 2: FILTRADO TOP 20 Y FEATURE ENGINEERING
# ==========================================
print("Calculando concentración de deuda para el Top 20...")

# Identificar las 20 entidades que mayor volumen de deuda capturan en el sistema formal
top_20_calculado = joined_con_nombres.group_by(["cod_entidad", "nombre_entidad", "grupo_entidad"]).agg(
    pl.col("deuda_total").sum().alias("volumen_deuda_total")
).sort("volumen_deuda_total", descending=True).limit(20)

df_top20_ranking = top_20_calculado.collect().to_pandas()

# Filtrar la base maestra perezosa para conservar únicamente registros del Top 20
lista_codigos_top20 = df_top20_ranking["cod_entidad"].tolist()
base_filtrada_top20 = joined_con_nombres.filter(pl.col("cod_entidad").is_in(lista_codigos_top20))

# 3.1 Feature Engineering (Edades y Macro-Regiones Geográficas)
base_features = base_filtrada_top20.with_columns(
    pl.col("nacimiento_clean").str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias("fecha_nac_dt"),
    pl.col("codigo_postal").cast(pl.Int32, strict=False).fill_null(0).alias("cp_num")
).with_columns(
    ((pl.date(2026, 1, 1) - pl.col("fecha_nac_dt")).dt.total_days() / 365.25).floor().alias("edad")
).with_columns([
    pl.when(pl.col("edad") <= 25).then(pl.lit("Jóvenes (18-25)"))
    .when(pl.col("edad") <= 40).then(pl.lit("Adultos en Inserción (26-40)"))
    .when(pl.col("edad") <= 65).then(pl.lit("Adultos Consolidados (41-65)"))
    .otherwise(pl.lit("Adultos Mayores (>65)")).alias("Edad"),

    # 🗺️ LÓGICA DE MACRO-REGIONES (INDEC + AMBA)
    pl.when((pl.col("cp_num") >= 1000) & (pl.col("cp_num") <= 1499)).then(pl.lit("AMBA - CABA"))
    .when((pl.col("cp_num") >= 1600) & (pl.col("cp_num") <= 1999)).then(pl.lit("AMBA - Conurbano"))
    .when(pl.col("provincia").is_in(["01", "03", "05", "11", "19"])).then(pl.lit("Región Pampeana")) # BsAs, Cba, ER, SF, LP
    .when(pl.col("provincia").is_in(["02", "06", "08", "12", "13", "23"])).then(pl.lit("NOA")) # Cat, Juj, LR, SdE, Tuc, Sal
    .when(pl.col("provincia").is_in(["04", "14", "16", "17"])).then(pl.lit("NEA")) # Corr, Cha, For, Mis
    .when(pl.col("provincia").is_in(["07", "09", "10"])).then(pl.lit("Cuyo")) # Men, SJ, SL
    .when(pl.col("provincia").is_in(["15", "18", "20", "21", "22"])).then(pl.lit("Patagonia")) # Chu, Neu, RN, SC, TdF
    .otherwise(pl.lit("Sin Especificar")).alias("Geografia")
]).select([
    pl.col("cod_entidad").alias("cod_entidad"),
    pl.col("nombre_entidad").alias("nombre_entidad"),
    pl.col("grupo_entidad").alias("grupo_entidad"),
    pl.col("segmento").alias("segmento"),
    pl.col("deuda_total").alias("deuda_total"),
    pl.col("sexo_clean").alias("sexo_clean"),
    pl.col("Edad"),
    pl.col("Geografia")
]).drop_nulls()

print("Materializando base consolidada...")
df_eda = base_features.collect().to_pandas()

# --- IMPRESIÓN DE SUMMARY METRICS EXPLICATORIAS ---
print("\n" + "="*55)
print("📊 SUMMARY METRICS: UNIVERSO TOP 20 ENTIDADES (ARCA-BCRA)")
print("="*55)
print(f"Total de registros deudores en el Top 20: {len(df_eda):,}")
print(f"Total de deuda administrada por el Top 20: ${df_eda['deuda_total'].sum():,.2f}")
print("\nDistribución Geográfica:")
print(df_eda['Geografia'].value_counts(normalize=True).round(4) * 100)
print("="*55 + "\n")



Calculando concentración de deuda para el Top 20...
Materializando base consolidada...

📊 SUMMARY METRICS: UNIVERSO TOP 20 ENTIDADES (ARCA-BCRA)
Total de registros deudores en el Top 20: 3,329,995
Total de deuda administrada por el Top 20: $8,869,093,784.00

Distribución Geográfica:
Geografia
AMBA - Conurbano    26.26
Región Pampeana     25.79
NOA                 13.11
AMBA - CABA         10.97
NEA                  8.85
Cuyo                 8.37
Patagonia            5.63
Sin Especificar      1.02
Name: proportion, dtype: float64



## <span style="color: #2b7a78;">4. Visualizaciones Descriptivas del Mercado 📉</span>


In [12]:
# ==========================================
# 📈 PASO 3: BATERÍA DE GRÁFICOS EXPLORATORIOS
# ==========================================
print("Generando visualizaciones descriptivas...")

# Gráfico 1: Concentración del Volumen de Deuda por Entidad Real
fig1 = px.bar(
    df_top20_ranking,
    x='volumen_deuda_total',
    y='nombre_entidad',
    color='grupo_entidad',
    orientation='h',
    title='1. Concentración de Deuda Total - Top 20 Entidades Financieras',
    labels={'volumen_deuda_total': 'Volumen de Deuda Acumulada ($)', 'nombre_entidad': 'Entidad Financiera', 'grupo_entidad': 'Grupo Institucional'},
    color_discrete_sequence=px.colors.qualitative.Dark24
)
fig1.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_white', height=600)
fig1.show()

# Gráfico 2: Asimetría Geográfica de las carteras por Grupo Institucional
df_g4 = pd.crosstab(df_eda['grupo_entidad'], df_eda['Geografia'], normalize='index') * 100
fig4 = px.bar(
    df_g4.reset_index().melt(id_vars='grupo_entidad'),
    x='value', y='grupo_entidad', color='Geografia', orientation='h',
    title='2. Segmentación Geográfica de las Carteras por Grupo Institucional (%)',
    labels={'value': 'Porcentaje de la Cartera (%)', 'grupo_entidad': 'Grupo Institucional', 'Geografia': 'Región'},
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig4.update_layout(template='plotly_white')
fig4.show()

Generando visualizaciones descriptivas...


/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




## <span style="color: #2b7a78;">5. Preprocesamiento de Masas y Análisis de Correspondencias Múltiples (MCA) 🧠</span>
Eliminamos categorías atípicas para no sesgar el plano factorial, extraemos una muestra representativa de 250k deudores y corremos el MCA de la librería `prince`.


In [13]:
# ==========================================
# 🧠 PASO 4: PREPROCESAMIENTO Y MCA
# ==========================================
print("\n=======================================================")
print("🧮 DISTRIBUCIÓN DE MASAS (PREVIO AL FILTRADO)")
print("=======================================================")
for col in ['nombre_entidad', 'sexo_clean', 'Edad', 'Geografia']:
    print(f"--- {col} ---")
    distribucion = df_eda[col].value_counts(normalize=True) * 100
    print(distribucion.round(2).to_string(), "\n")

# 1. Eliminar outliers categóricos (Género X y categorías residuales)
df_mca_limpio = df_eda[df_eda['sexo_clean'].isin(['M', 'F'])].copy()

# 2. Seleccionar SOLO las variables con varianza real (Quitamos Escala_Deuda por tener 99% en Pequeño)
X_mca = df_mca_limpio[['nombre_entidad', 'sexo_clean', 'Edad', 'Geografia']].copy()
X_mca.columns = ['Entidad', 'Genero', 'Edad', 'Geografia']

# Tomar muestra controlada de 250k para no saturar RAM
df_mca_sample = X_mca.sample(n=min(250000, len(X_mca)), random_state=42)

print("\nEntrenando modelo MCA...")
mca = prince.MCA(n_components=2, n_iter=10, random_state=42)
mca = mca.fit(df_mca_sample)

v1, v2 = mca.percentage_of_variance_[0], mca.percentage_of_variance_[1]
print(f"¡MCA Finalizado! Varianza retenida (Inercia cruda): Dim1={v1:.2f}% | Dim2={v2:.2f}%")

# Extraer coordenadas limpias
coords = mca.column_coordinates(df_mca_sample).copy()
coords.columns = ['Dim_1', 'Dim_2']
coords['Atributo_Original'] = coords.index

def limpiar_nombre(attr):
    prefijos = ['Entidad_', 'Edad_', 'Geografia_', 'Genero_']
    for p in prefijos:
        if str(attr).startswith(p): return str(attr).replace(p, '', 1)
    return str(attr)

def clasificar_variable(attr):
    attr_str = str(attr)
    if attr_str.startswith('Entidad_'): return 'Top 20 Bancos/Fintech'
    if attr_str.startswith('Edad_'): return 'Rango Etario'
    if attr_str.startswith('Geografia_'): return 'Región Geográfica'
    if attr_str.startswith('Genero_'): return 'Género'
    return 'Otro'

coords['Categoria'] = coords['Atributo_Original'].apply(limpiar_nombre)
coords['Tipo'] = coords['Atributo_Original'].apply(clasificar_variable)

# Gráfico 5: El Mapa Perceptual Definitivo
fig5 = px.scatter(
    coords, x='Dim_1', y='Dim_2', color='Tipo', text='Categoria',
    title='Mapa Perceptual MCA - Top 20 Entidades y Atributos Demográficos',
    labels={'Dim_1': f"Dimensión 1 ({v1:.2f}%)", 'Dim_2': f"Dimensión 2 ({v2:.2f}%)"}
)
fig5.update_traces(textposition='top center', marker=dict(size=13, opacity=0.85, line=dict(width=1, color='DarkSlateGrey')))
fig5.update_layout(template='plotly_white', width=1300, height=850)
fig5.show()




🧮 DISTRIBUCIÓN DE MASAS (PREVIO AL FILTRADO)
--- nombre_entidad ---
nombre_entidad
MERCADOLIBRE S.R.L.                                           21.42
TARJETA NARANJA S.A.                                          14.62
BANCO DE LA NACION ARGENTINA                                   8.34
BANCO DE GALICIA Y BUENOS AIRES S.A.                           7.73
BANCO BBVA ARGENTINA S.A.                                      7.38
BANCO MACRO S.A.                                               7.07
BANCO DE LA PROVINCIA DE BUENOS AIRES                          6.80
BANCO SANTANDER ARGENTINA S.A.                                 6.78
NARANJA DIGITAL COMPANIA FINANCIERA S.A.U.                     4.71
INDUSTRIAL AND COMMERCIAL BANK OF CHINA (ARGENTINA) S.A.U.     2.26
BANCO PATAGONIA S.A.                                           1.98
BANCO DE LA PROVINCIA DE CORDOBA S.A.                          1.96
BANCO SUPERVIELLE S.A.                                         1.75
NUEVO BANCO DE SANTA FE SOCIEDAD

## <span style="color: #2b7a78;">6. Exportación del Grafo a HTML 🚀</span>


In [14]:
# Guardar como HTML interactivo
archivo_html = "Mapa_MCA_Top20_Regiones.html"
fig5.write_html(archivo_html)
print(f"¡Gráfico interactivo guardado en: {archivo_html}!")



¡Gráfico interactivo guardado en: Mapa_MCA_Top20_Regiones.html!


In [16]:
# ==========================================
# 🌐 PASO 7: HOSTEAR EL HTML EN UNA PÁGINA APARTE
# ==========================================
import threading
import http.server
import socketserver
import os
from google.colab import output

# Definimos un puerto libre
PORT = 8081

# Nos aseguramos de que el servidor busque los archivos donde se guardó el HTML.
# Si lo guardaste en tu Drive, podés cambiar esta ruta al directorio del Drive.
# En este caso, asumimos que el archivo "Mapa_MCA_Top20_Regiones.html" quedó en la carpeta local de Colab.
os.chdir('/content')

def iniciar_servidor():
    # Configuramos el servidor HTTP básico de Python
    Handler = http.server.SimpleHTTPRequestHandler
    # Permitimos reutilizar el puerto para evitar errores de "Address already in use"
    socketserver.TCPServer.allow_reuse_address = True

    with socketserver.TCPServer(("", PORT), Handler) as httpd:
        print(f"\n[INFO] Servidor web local iniciado exitosamente en el puerto {PORT}.")
        httpd.serve_forever()

# Iniciamos el servidor en un Thread (hilo) secundario.
# Esto es CLAVE para que Colab no se quede "pensando" infinitamente en esta celda.
threading.Thread(target=iniciar_servidor, daemon=True).start()

print("Generando enlace seguro de visualización...")
# Esta función mágica de Colab abre una nueva pestaña del navegador conectada al puerto local
output.serve_kernel_port_as_window(PORT, path='/Mapa_MCA_Top20_Regiones.html')


[INFO] Servidor web local iniciado exitosamente en el puerto 8081.
Generando enlace seguro de visualización...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>